In [1]:
import os 

In [2]:
os.chdir('..')

In [3]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [28]:
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.constants import CONFIG_PATH
from sklearn.metrics import mean_absolute_error
from dataclasses import dataclass
import glob
from datetime import datetime
import mlflow
import shutil
import pandas as pd
import pickle
import json
import numpy as np
from pathlib import Path

In [29]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    test_data_path        : Path
    model_dir             : Path
    champion_path         : Path
    report_dir            : Path
    features              : list[str]
    target_column         : str
    baseline_mae          : float
    improvement_threshold : float
    champion_threshold    : float
    spike_threshold       : float
    mlflow_experiment     : str
    mlflow_tracking_uri   : str

In [30]:
class Config_manager:

    def __init__(self ,config = CONFIG_PATH):
        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:

        config = self.config.model_evaluation

        create_directories([config.report_dir])

        return ModelEvaluationConfig(
            test_data_path        = Path(config.test_data_path),
            model_dir             = Path(config.model_dir),
            champion_path         = Path(config.champion_path),
            report_dir            = Path(config.report_dir),
            features              = list(config.features),
            target_column         = config.target_column,
            baseline_mae          = float(config.baseline_mae),
            improvement_threshold = float(config.improvement_threshold),
            champion_threshold    = float(config.champion_threshold),
            spike_threshold       = float(config.spike_threshold),
            mlflow_experiment     = config.mlflow.experiment_name,
            mlflow_tracking_uri   = config.mlflow.tracking_uri
        )

In [ ]:
class Model_evalulation:

    def __init__(self,config : ModelEvaluationConfig):
        self.config = config
        self.test = self.read_data()
        self.model  = self._load_latest_model()

    
    def read_data(self):

        try:
            files = glob.glob(os.path.join(self.config.test_data_path , "test_*.csv"))
            if not files:
                raise FileNotFoundError(
                    f"No test file found in {self.config.test_data_path}"
                )
            latest = max(files, key=os.path.getmtime)
            df     = pd.read_csv(latest)

            logger.info(f"Test rows : {len(df)}")
            return df
        except FileNotFoundError as e:
            logger.error(f"Test file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to read test data: {str(e)}")
            raise 

    def _load_latest_model(self):
        try:
            files = glob.glob(
                os.path.join(self.config.model_dir, "model_*.pkl")
            )

            if not files:
                raise FileNotFoundError(
                    f"No trained model found in {self.config.model_dir}"
                )

            latest = max(files, key=os.path.getmtime)
            logger.info(f"Loading model : {latest}")

            with open(latest, "rb") as f:
                model = pickle.load(f)

            logger.info("Model loaded successfully")
            return model, latest

        except FileNotFoundError as e:
            logger.error(f"Model file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to load model: {str(e)}")
            raise
        
    def _prepare_xy(self):
        try:
            X = self.test[self.config.features]
            y = self.test[self.config.target_column]
            return X, y

        except Exception as e:
            logger.error(f"Failed to prepare X and y: {str(e)}")
            raise
    
    def _smape(self, actual, predicted):
        try:
            return float(
                100 * np.mean(
                    2 * np.abs(predicted - actual) /
                    (np.abs(actual) + np.abs(predicted) + 1e-8)
                )
            )
        except Exception as e:
            logger.error(f"SMAPE calculation failed: {str(e)}")
            raise
    
    def evaluate_baseline(self):
        try:
            logger.info("")
            logger.info("STEP 1 - BASELINE EVALUATION")
            logger.info("-" * 50)

            X, y         = self._prepare_xy()
            baseline_pred = X["lag_1"]
            baseline_mae  = float(mean_absolute_error(y, baseline_pred))

            logger.info(f"Baseline MAE : {baseline_mae:.6f}")
            logger.info("PASSED - Baseline evaluated")

            return baseline_mae

        except Exception as e:
            logger.error(f"Baseline evaluation failed: {str(e)}")
            raise
        
    def evaluate_model(self):
        try:
            logger.info("")
            logger.info("STEP 2 - MODEL EVALUATION ON TEST SET")
            logger.info("-" * 50)

            model, _      = self.model
            X, y          = self._prepare_xy()
            pred          = model.predict(X)

            test_mae      = float(mean_absolute_error(y, pred))
            test_smape    = self._smape(y.values, pred)

            logger.info(f"Test MAE   : {test_mae:.6f}")
            logger.info(f"Test SMAPE : {test_smape:.2f}%")
            logger.info("PASSED - Model evaluated")

            return test_mae, test_smape

        except Exception as e:
            logger.error(f"Model evaluation failed: {str(e)}")
            raise
    
    def deployment_decision(self, test_mae, baseline_mae):
        try:
            logger.info("")
            logger.info("STEP 3 - DEPLOYMENT DECISION")
            logger.info("-" * 50)

            improvement = (baseline_mae - test_mae) / baseline_mae * 100

            logger.info(f"Baseline MAE  : {baseline_mae:.6f}")
            logger.info(f"Model MAE     : {test_mae:.6f}")
            logger.info(f"Improvement   : {improvement:.2f}%")
            logger.info(f"Threshold     : {self.config.improvement_threshold * 100:.0f}%")

            if improvement < self.config.improvement_threshold * 100:
                logger.error("FAILED - Model does not beat baseline threshold")
                logger.error("Model REJECTED - will not be deployed")
                return False, improvement

            logger.info("PASSED - Model beats baseline threshold")
            return True, improvement

        except Exception as e:
            logger.error(f"Deployment decision failed: {str(e)}")
            raise
    
    def champion_comparison(self,test_mae):
        try:
            if not os.path.exists(self.config.champion_path):

                return True, None

            with open(self.config.champion_path,'rb') as f:
                champion = pickle.load(f)
            
            X ,y = self._prepare_xy()
            champion_pred = champion.predict(X)
            champion_mae  = float(mean_absolute_error(y, champion_pred))
            improvement = (champion_mae - test_mae) / champion_mae * 100

            logger.info(f"Champion MAE   : {champion_mae:.6f}")
            logger.info(f"Challenger MAE : {test_mae:.6f}")
            logger.info(f"Improvement    : {improvement:.2f}%")
            logger.info(f"Threshold      : {self.config.champion_threshold * 100:.0f}%")

            if improvement > self.config.champion_threshold * 100:
                logger.info("PASSED - Challenger beats champion")
                return True, champion_mae
            
            logger.warning("WARNING - Challenger does not beat champion")
            logger.warning("Champion stays in Production")
            return False, champion_mae

        except Exception as e:
            logger.error(f"Champion comparison failed: {str(e)}")
            raise
    
    def promote_to_champion(self):
        try:
            logger.info("")
            logger.info("STEP 5 - PROMOTING TO CHAMPION")
            logger.info("-" * 50)

            _, model_path = self.model

            shutil.copy(model_path, self.config.champion_path)

            logger.info(f"Source      : {model_path}")
            logger.info(f"Destination : {self.config.champion_path}")
            logger.info("PASSED - Model promoted to champion")

        except Exception as e:
            logger.error(f"Promotion failed: {str(e)}")
            raise
    
    def save_report(self, report: dict):
        try:
            os.makedirs(self.config.report_dir, exist_ok=True)
            timestamp   = datetime.now().strftime("%Y_%m_%d_%H_%M")
            report_path = os.path.join(
                self.config.report_dir,
                f"evaluation_{timestamp}.json"
            )

            with open(report_path, "w") as f:
                json.dump(report, f, indent=2, default=str)

            logger.info(f"Report saved  {report_path}")
            return report_path

        except Exception as e:
            logger.error(f"Failed to save report: {str(e)}")
            raise
        
    def log_to_mlflow(self, report: dict, report_path: str):
        try:
            logger.info("")
            logger.info("STEP 6 - LOGGING TO MLFLOW")
            logger.info("-" * 50)

            mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
            mlflow.set_experiment(self.config.mlflow_experiment)

            with mlflow.start_run():

                mlflow.log_metric("test_mae",       report["test_mae"])
                mlflow.log_metric("test_smape",     report["test_smape"])
                mlflow.log_metric("baseline_mae",   report["baseline_mae"])
                mlflow.log_metric("improvement_pct",report["improvement_pct"])

                if report["champion_mae"] is not None:
                    mlflow.log_metric("champion_mae", report["champion_mae"])

                mlflow.set_tag("stage",          "Production" if report["promoted"] else "Archived")
                mlflow.set_tag("promoted",       str(report["promoted"]))
                mlflow.set_tag("beats_baseline", str(report["beats_baseline"]))
                mlflow.set_tag("evaluated_at",   report["evaluated_at"])

                mlflow.log_artifact(report_path)

            logger.info("PASSED - MLflow logging complete")

        except Exception as e:
            logger.error(f"MLflow logging failed: {str(e)}")
            raise

        
    def run(self):
        model_obj    = None
        champion_obj = None

        try:
            logger.info("=" * 50)
            logger.info("MODEL EVALUATION PIPELINE STARTED")
            logger.info("=" * 50)

            baseline_mae              = self.evaluate_baseline()
            test_mae, test_smape      = self.evaluate_model()
            beats_baseline, improvement = self.deployment_decision(test_mae, baseline_mae)

            report = {
                "evaluated_at"   : datetime.now().isoformat(),
                "test_mae"       : test_mae,
                "test_smape"     : test_smape,
                "baseline_mae"   : baseline_mae,
                "improvement_pct": improvement,
                "champion_mae"   : None,
                "beats_baseline" : beats_baseline,
                "promoted"       : False
            }

            promote, champion_mae    = self.champion_comparison(test_mae)
            report["champion_mae"]   = champion_mae

            if promote:
                self.promote_to_champion()
                report["promoted"] = True

            report_path = self.save_report(report)
            self.log_to_mlflow(report, report_path)

            logger.info("=" * 50)
            logger.info("MODEL EVALUATION COMPLETE")
            logger.info(f"Promoted : {report['promoted']}")
            logger.info("=" * 50)

            return report["promoted"]

        except Exception as e:
            logger.error(f"Model evaluation pipeline failed: {str(e)}")
            raise

        finally:
            self.test  = None
            self.model = None
            import gc
            gc.collect()
            logger.info("Memory cleared")

In [73]:
xc = Config_manager()
xc = xc.get_model_evaluation_config()
xc = Model_evalulation(xc)
xc.run()

[2026-06-15 16:07:04,362: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-15 16:07:04,372: INFO: common: Directory created (or already exists) at: artifacts]


[2026-06-15 16:07:04,405: INFO: common: Directory created (or already exists) at: artifacts/models_directory/evaluation_results/]
[2026-06-15 16:07:04,450: INFO: 1553982202: Test rows : 8122]
[2026-06-15 16:07:04,450: INFO: 1553982202: Loading model : artifacts\models_directory\model_2026_06_15_15_39.pkl]
[2026-06-15 16:07:04,540: INFO: 1553982202: Model loaded successfully]
[2026-06-15 16:07:04,540: INFO: 1553982202: ==================================================]
[2026-06-15 16:07:04,540: INFO: 1553982202: MODEL EVALUATION PIPELINE STARTED]
[2026-06-15 16:07:04,540: INFO: 1553982202: ==================================================]
[2026-06-15 16:07:04,540: INFO: 1553982202: ]
[2026-06-15 16:07:04,540: INFO: 1553982202: STEP 1 - BASELINE EVALUATION]
[2026-06-15 16:07:04,540: INFO: 1553982202: --------------------------------------------------]
[2026-06-15 16:07:04,572: INFO: 1553982202: Baseline MAE : 0.020326]
[2026-06-15 16:07:04,580: INFO: 1553982202: PASSED - Baseline eval

AttributeError: 'Model_evalulation' object has no attribute 'champion_comparison'

In [ ]:
(0.018283242290870402, 2.293507581733747)
class Model_evalulation:

    def __init__(self,config : ModelEvaluationConfig):
        self.config = config
        self.read_data()

    
    def read_data(self):

        try:
            files = glob.glob(os.path.join(self.config.test_data_path , "test_*.csv"))
            if not files:
                raise FileNotFoundError(
                    f"No test file found in {self.config.test_data_path}"
                )
            latest = max(files, key=os.path.getmtime)
            df     = pd.read_csv(latest)

            logger.info(f"Test rows : {len(df)}")
            return df
        except FileNotFoundError as e:
            logger.error(f"Test file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to read test data: {str(e)}")
            raise 

    def _load_latest_model(self):
        try:
            files = glob.glob(
                os.path.join(self.config.model_dir, "model_*.pkl")
            )

            if not files:
                raise FileNotFoundError(
                    f"No trained model found in {self.config.model_dir}"
                )

            latest = max(files, key=os.path.getmtime)
            logger.info(f"Loading model : {latest}")

            with open(latest, "rb") as f:
                model = pickle.load(f)

            logger.info("Model loaded successfully")
            return model, latest

        except FileNotFoundError as e:
            logger.error(f"Model file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to load model: {str(e)}")
            raise
        
    def _prepare_xy(self):
        try:
            X = self.test[self.config.features]
            y = self.test[self.config.target_column]
            return X, y

        except Exception as e:
            logger.error(f"Failed to prepare X and y: {str(e)}")
            raise
    
    def _smape(self, actual, predicted):
        try:
            return float(
                100 * np.mean(
                    2 * np.abs(predicted - actual) /
                    (np.abs(actual) + np.abs(predicted) + 1e-8)
                )
            )
        except Exception as e:
            logger.error(f"SMAPE calculation failed: {str(e)}")
            raise
    
    def evaluate_baseline(self):
        try:
            logger.info("")
            logger.info("STEP 1 - BASELINE EVALUATION")
            logger.info("-" * 50)

            X, y         = self._prepare_xy()
            baseline_pred = X["lag_1"]
            baseline_mae  = float(mean_absolute_error(y, baseline_pred))

            logger.info(f"Baseline MAE : {baseline_mae:.6f}")
            logger.info("PASSED - Baseline evaluated")

            return baseline_mae

        except Exception as e:
            logger.error(f"Baseline evaluation failed: {str(e)}")
            raise
    
    def evaluate_model(self):
        try:
            logger.info("")
            logger.info("STEP 2 - MODEL EVALUATION ON TEST SET")
            logger.info("-" * 50)

            model, _      = self.model
            X, y          = self._prepare_xy()
            pred          = model.predict(X)

            test_mae      = float(mean_absolute_error(y, pred))
            test_smape    = self._smape(y.values, pred)

            logger.info(f"Test MAE   : {test_mae:.6f}")
            logger.info(f"Test SMAPE : {test_smape:.2f}%")
            logger.info("PASSED - Model evaluated")

            return test_mae, test_smape

        except Exception as e:
            logger.error(f"Model evaluation failed: {str(e)}")
            raise

    def deployment_decision(self, test_mae, baseline_mae):
        try:
            logger.info("")
            logger.info("STEP 3 - DEPLOYMENT DECISION")
            logger.info("-" * 50)

            improvement = (baseline_mae - test_mae) / baseline_mae * 100

            logger.info(f"Baseline MAE  : {baseline_mae:.6f}")
            logger.info(f"Model MAE     : {test_mae:.6f}")
            logger.info(f"Improvement   : {improvement:.2f}%")
            logger.info(f"Threshold     : {self.config.improvement_threshold * 100:.0f}%")

            if improvement < self.config.improvement_threshold * 100:
                logger.error("FAILED - Model does not beat baseline threshold")
                logger.error("Model REJECTED - will not be deployed")
                return False, improvement

            logger.info("PASSED - Model beats baseline threshold")
            return True, improvement

        except Exception as e:
            logger.error(f"Deployment decision failed: {str(e)}")
            raise

    def champion_(self,test_mae):
        try:
            if not os.path.exists(self.config.champion_path):

                return True, None

            with open(self.config.champion_path,'rb') as f:
                champion = pickle.load(f)
            
            X ,y = self._prepare_xy()
            champion_pred = champion.predict(X)
            champion_mae  = float(mean_absolute_error(y, champion_pred))
            improvement = (champion_mae - test_mae) / champion_mae * 100

            logger.info(f"Champion MAE   : {champion_mae:.6f}")
            logger.info(f"Challenger MAE : {test_mae:.6f}")
            logger.info(f"Improvement    : {improvement:.2f}%")
            logger.info(f"Threshold      : {self.config.champion_threshold * 100:.0f}%")

            if improvement > self.config.champion_threshold * 100:
                logger.info("PASSED - Challenger beats champion")
                return True, champion_mae
            
            logger.warning("WARNING - Challenger does not beat champion")
            logger.warning("Champion stays in Production")
            return False, champion_mae

        except Exception as e:
            logger.error(f"Champion comparison failed: {str(e)}")
            raise

    def promote_to_champion(self):
        try:
            logger.info("")
            logger.info("STEP 5 - PROMOTING TO CHAMPION")
            logger.info("-" * 50)

            _, model_path = self.model

            shutil.copy(model_path, self.config.champion_path)

            logger.info(f"Source      : {model_path}")
            logger.info(f"Destination : {self.config.champion_path}")
            logger.info("PASSED - Model promoted to champion")

        except Exception as e:
            logger.error(f"Promotion failed: {str(e)}")
            raise
    
    def save_report(self, report: dict):
        try:
            os.makedirs(self.config.report_dir, exist_ok=True)
            timestamp   = datetime.now().strftime("%Y_%m_%d_%H_%M")
            report_path = os.path.join(
                self.config.report_dir,
                f"evaluation_{timestamp}.json"
            )

            with open(report_path, "w") as f:
                json.dump(report, f, indent=2, default=str)

            logger.info(f"Report saved  {report_path}")
            return report_path

        except Exception as e:
            logger.error(f"Failed to save report: {str(e)}")
            raise
    
    def log_to_mlflow(self, report: dict, report_path: str):
        try:
            logger.info("")
            logger.info("STEP 6 - LOGGING TO MLFLOW")
            logger.info("-" * 50)

            mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
            mlflow.set_experiment(self.config.mlflow_experiment)

            with mlflow.start_run():

                mlflow.log_metric("test_mae",       report["test_mae"])
                mlflow.log_metric("test_smape",     report["test_smape"])
                mlflow.log_metric("baseline_mae",   report["baseline_mae"])
                mlflow.log_metric("improvement_pct",report["improvement_pct"])

                if report["champion_mae"] is not None:
                    mlflow.log_metric("champion_mae", report["champion_mae"])

                mlflow.set_tag("stage",          "Production" if report["promoted"] else "Archived")
                mlflow.set_tag("promoted",       str(report["promoted"]))
                mlflow.set_tag("beats_baseline", str(report["beats_baseline"]))
                mlflow.set_tag("evaluated_at",   report["evaluated_at"])

                mlflow.log_artifact(report_path)

            logger.info("PASSED - MLflow logging complete")

        except Exception as e:
            logger.error(f"MLflow logging failed: {str(e)}")
            raise
    
    def run(self):
        model_obj    = None
        champion_obj = None

        try:
            logger.info("=" * 50)
            logger.info("MODEL EVALUATION PIPELINE STARTED")
            logger.info("=" * 50)

            baseline_mae              = self.evaluate_baseline()
            test_mae, test_smape      = self.evaluate_model()
            beats_baseline, improvement = self.deployment_decision(test_mae, baseline_mae)

            report = {
                "evaluated_at"   : datetime.now().isoformat(),
                "test_mae"       : test_mae,
                "test_smape"     : test_smape,
                "baseline_mae"   : baseline_mae,
                "improvement_pct": improvement,
                "champion_mae"   : None,
                "beats_baseline" : beats_baseline,
                "promoted"       : False
            }

            promote, champion_mae    = self.champion_comparison(test_mae)
            report["champion_mae"]   = champion_mae

            if promote:
                self.promote_to_champion()
                report["promoted"] = True

            report_path = self.save_report(report)
            self.log_to_mlflow(report, report_path)

            logger.info("=" * 50)
            logger.info("MODEL EVALUATION COMPLETE")
            logger.info(f"Promoted : {report['promoted']}")
            logger.info("=" * 50)

            return report["promoted"]

        except Exception as e:
            logger.error(f"Model evaluation pipeline failed: {str(e)}")
            raise

        finally:
            self.test  = None
            self.model = None
            import gc
            gc.collect()
            logger.info("Memory cleared")